# CVM Free Cash Flow Pipeline

Extracts the historical **free cash flow (FCF)** of Brazilian listed companies from the
open data portal of the CVM (Comissão de Valores Mobiliários), as the first step of a
discounted cash flow (DCF) valuation.

**Pipeline overview**

1. **Sources**: list the yearly DFP zip files on the CVM website, download them (with a local cache) and read the cash flow statement.
2. **Cleaning**: keep the latest fiscal year and the most recent document of each company, drop duplicates and normalize the currency scale.
3. **Metrics**: compute operating cash flow (CFO), capex and FCF for each company and year.
4. **Registry**: add sector and listing information from the CVM company registry and flag financial and investable companies.

All values are in **thousands of reais**. Capex is negative (cash out), so `FCF = CFO + CAPEX`.

> Sections 1 and 2 only define things. Section 3 is the first one that downloads data.

## 1. Setup

### 1.1 Libraries

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import zipfile
import io
from datetime import date
from pathlib import Path

### 1.2 Configuration

Everything that may need adjusting lives here, in one place:

- **Paths and URLs.** `DATA_DIR` is the local cache for the downloaded zip files (ignored by Git).
- **Capex rule.** A line of the investing section (`6.02`) counts as capex if its description matches `CAPEX_TERMS` and does **not** match `EXCLUDE_TERMS`. The list describes what capex *is* (an allowlist) on purpose: an unknown line is left out instead of being counted, so errors stay small.
- **Financial sectors.** Banks, insurers and similar companies are flagged, since free cash flow is not a meaningful measure for them.

The terms are regular expressions: `s?` makes the `s` optional and `.` matches any character (useful for accents).

In [2]:
DATA_DIR = Path("../data")
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"
CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
CAPEX_TERMS = [
    "imobiliz",
    "intang",
    "software",
    "ativos? de contrato",
    "ativos? contratua",
    "infraestrutura",
    "ativos? da concess",
    "servi.o p.blico",
    "biol",
    "propriedades? para investimento",
]

EXCLUDE_TERMS = [
    "venda",
    "aliena",
    "baixa",
    "receb",
    "indeniza",
    "adiantamento",
    "transf",
    "decr.scimo",
]

CAPEX_KEYWORDS = "|".join(CAPEX_TERMS)
EXCLUDE_KEYWORDS = "|".join(EXCLUDE_TERMS)
FINANCIAL_SECTORS = [
    "Bancos",
    "Intermediação Financeira",
    "Crédito Imobiliário",
    "Arrendamento Mercantil",
    "Securitização de Recebíveis",
    "Factoring",
    "Seguradoras e Corretoras",
]
FINANCIAL_KEYWORDS = "|".join(FINANCIAL_SECTORS)
pd.set_option("display.max_rows", None)


## 2. Functions

### 2.1 Data sources

- `find_zips` reads the CVM directory page and maps each year to its zip URL (web scraping).
- `complete_years` drops years whose files are still incomplete: the current year and, before April, the previous one (companies have until the end of March to file).
- `get_file` downloads a file into memory, caching it in `DATA_DIR`. Use `use_cache=False` for files that change often, such as the company registry.
- `open_csv` reads one statement from a zip, e.g. the consolidated cash flow statement (`DFC_MI`, `con`).

In [3]:
def find_zips(url):
    """Return a dictionary mapping each year to its zip URL."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tags = soup.find_all("a")
    zip_dict = {}
    for tag in tags:
        href = tag.get("href")
        if not href:
            continue
        if href.endswith(".zip"):
            year = int(href.removesuffix(".zip")[-4:])
            link = urljoin(url, href)
            zip_dict[year] = link
    return zip_dict

def complete_years(zip_links):
    """Return only the years whose DFP files are complete.

    Excludes the current year and, before April, also the previous
    year, since companies have until the end of March to file.
    """
    today = date.today()
    complete_year_links = {}
    for year, link in zip_links.items():
        if (year == today.year - 1 and today.month < 4) or (year >= today.year):
            continue
        complete_year_links[year] = link
    return complete_year_links

def get_file(link, use_cache=True):
    """Return a file from the given URL as an in-memory buffer.

    By default, the file is cached in DATA_DIR and only downloaded if it
    is not there yet. With use_cache=False, it is always downloaded and
    never saved.
    """
    if not use_cache:
        response = requests.get(link, timeout=60)
        response.raise_for_status()
        return io.BytesIO(response.content)
    DATA_DIR.mkdir(exist_ok=True)
    file_name = link.split("/")[-1]
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        response = requests.get(link, timeout=60)
        response.raise_for_status()
        file_path.write_bytes(response.content)
    return io.BytesIO(file_path.read_bytes())

def open_csv(zip_file, statement, basis, year, doc_type="dfp"):
    """Read a statement CSV from a CVM zip (DFP or ITR) and return a DataFrame."""
    with zipfile.ZipFile(zip_file) as z:
        csv_name = f"{doc_type}_cia_aberta_{statement}_{basis}_{year}.csv"
        df = pd.read_csv(z.open(csv_name), sep=";", encoding="latin-1")
    return df

### 2.2 Cleaning

The raw CVM files need four corrections before any calculation. Each one was found in the real data:

| Problem | Example | Fix |
|---|---|---|
| Each file repeats the previous year as a comparative | Every account appears as `ÚLTIMO` and `PENÚLTIMO` | Keep `ÚLTIMO` only |
| Fully duplicated rows | 320 identical rows in 2025 (FGR Incorporações, VLI Multimodal) | `drop_duplicates` |
| Values in units instead of thousands | 7 companies in 2025 report in `UNIDADE` | Divide by 1,000 |
| More than one document per company | Estrela (8427) filed two DFPs in 2011: a 3-month one and the full year | Keep the latest `DT_REFER`, then the latest `VERSAO` |

`DT_REFER` is text, but `max()` still works because dates are in `YYYY-MM-DD` format, where alphabetical and chronological order match.

In [4]:
def split_by_company(df):
    """Clean a CVM statement and split it by company.

    Keeps only the most recent document of each company
    (latest DT_REFER, then latest VERSAO), drops duplicated rows and
    converts values to thousands of reais. Returns a dictionary
    mapping each CD_CVM to its DataFrame.
    """
    df_clean = df[(df["ORDEM_EXERC"] == "ÚLTIMO") & (~df.duplicated())]
    is_unit = df_clean["ESCALA_MOEDA"] == "UNIDADE"
    df_clean.loc[is_unit, "VL_CONTA"] /= 1000
    df_clean.loc[is_unit, "ESCALA_MOEDA"] = "MIL"
    companies_data = {}
    for code, group in df_clean.groupby("CD_CVM"):
        latest_date = group["DT_REFER"].max()
        group = group[group["DT_REFER"] == latest_date]
        latest_version = group["VERSAO"].max()
        group = group[group["VERSAO"] == latest_version]
        companies_data[code] = group
    return companies_data

### 2.3 Cash flow metrics

- **CFO** is the standardized account `6.01` (net cash from operating activities). `find_cfo` returns `NaN` when the account is missing and raises an error when it appears more than once, instead of silently summing duplicates. This check is what revealed the Estrela case above.
- **Capex** has no standard account: each company names its own lines inside `6.02`. `find_capex` sums the lines whose description matches the capex terms and not the exclusion terms. Beyond fixed and intangible assets, the terms cover concession contract assets (utilities), biological assets (pulp, agribusiness) and investment properties (shopping malls).

Validation: for Cemig (2453) in 2025, the rule finds R$ 6.46 billion, against R$ 6.63 billion of "investimentos realizados" in the company's investor relations spreadsheet. The difference is expected: the cash flow statement is on a cash basis and the release on an accrual basis.

In [5]:
def find_capex(company):
    """Return the total capex of one company.

    Values are in thousands of reais (negative = cash out).
    """
    investment = company[company["CD_CONTA"].str.startswith("6.02.")]
    is_asset = investment["DS_CONTA"].str.contains(CAPEX_KEYWORDS, case=False)
    is_sale = investment["DS_CONTA"].str.contains(EXCLUDE_KEYWORDS, case=False)
    capex = investment[is_asset & ~is_sale]
    return capex["VL_CONTA"].sum()

def find_cfo(company):
    """Return the operating cash flow (account 6.01) of one company.

    Values are in thousands of reais. Returns NaN if the account is
    missing and raises ValueError if it appears more than once.
    """
    is_cfo = company["CD_CONTA"] == "6.01"
    cfo = company[is_cfo]
    cfo_value = cfo["VL_CONTA"]
    if len(cfo_value) == 0:
        return float("nan")
    elif len(cfo_value) == 1:
        return cfo_value.iloc[0]
    else:
        code = company["CD_CVM"].iloc[0]
        raise ValueError(f"Company {code} has {len(cfo_value)} rows for account 6.01")

### 2.4 Building the history

`build_fcf_table` computes the metrics for every company in one year, and `build_fcf_history` repeats the whole pipeline for each year and stacks the results into one table (one row per company and year).

`YEAR` is the year of the CVM file. For companies with a non-calendar fiscal year (e.g. ending in March), it refers to the fiscal year ending in that year.

In [6]:
def build_fcf_table(companies_data, year):
    """Build a table with the free cash flow of each company in one year.

    Returns a DataFrame with one row per company and the columns
    CD_CVM, DENOM_CIA, YEAR, CFO, CAPEX and FCF. Values are in
    thousands of reais; FCF = CFO + CAPEX (capex is negative).
    """
    rows = []
    for code, group in companies_data.items():
        cfo = find_cfo(group)
        capex = find_capex(group)
        rows.append({
            "CD_CVM": code, "DENOM_CIA": group["DENOM_CIA"].iloc[0], "YEAR": year,
            "CFO": cfo, "CAPEX": capex, "FCF": cfo + capex})
    return pd.DataFrame(rows)

def build_fcf_history(dfp_links):
    """Build the free cash flow history of all companies.

    Takes a dictionary mapping each year to its DFP zip URL and returns
    a single DataFrame with one row per company and year, with the
    columns CD_CVM, DENOM_CIA, YEAR, CFO, CAPEX and FCF (in thousands
    of reais).
    """
    tables = []
    for year, link in dfp_links.items():
        print(f"Processing {year}...")
        zip_data = get_file(link)
        df = open_csv(zip_data, "DFC_MI", "con", year, "dfp")
        companies_data = split_by_company(df)
        fcf_table = build_fcf_table(companies_data, year)
        tables.append(fcf_table)
    return pd.concat(tables, ignore_index=True)

## 3. Load data

This is the only section that accesses the internet. The first run downloads one zip per year (about 13 MB each) into `DATA_DIR`; later runs read them from disk. To refresh a year, delete its file from the `data/` folder.

The company registry changes often, so it is always downloaded (`use_cache=False`).

In [7]:
zip_links = find_zips(DFP_URL)
dfp_links = complete_years(zip_links)
history = build_fcf_history(dfp_links)
cad = pd.read_csv(get_file(CAD_URL, use_cache=False), sep=";", encoding="latin-1")

Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...


## 4. Company registry

The CVM registry adds the sector and listing status of each company.

**Repeated rows.** A company traded in more than one market (e.g. `BOLSA` and `BALCÃO ORGANIZADO`) appears once per market. Before merging, only one row per company is kept, and sorting `TP_MERC` in descending order makes the `BOLSA` row the one that is kept. Without this step, the merge would duplicate rows in the history.

In [8]:
cad_small = cad[["CD_CVM", "SETOR_ATIV", "SIT", "CATEG_REG", "TP_MERC", "SIT_EMISSOR"]]
cad_small = cad_small.sort_values("TP_MERC", ascending=False)
cad_small = cad_small.drop_duplicates(subset="CD_CVM")

### 4.1 Flags

Companies are **flagged, not removed**, so the history stays complete. Removing companies that are no longer listed would keep only the survivors and make any historical comparison look better than reality (survivorship bias).

- `IS_FINANCIAL`: the sector is one of `FINANCIAL_SECTORS`, including the holding company versions ("Emp. Adm. Part. - Bancos", ...).
- `IS_INVESTABLE`: the company could be analyzed and bought **today**: active registration, category A (allowed to have traded shares), listed on the exchange, operating normally (not in judicial recovery) and not financial. Since the registry describes the present, this flag refers to today for every year of the history.

The merge uses `how="left"` so that companies missing from the registry stay in the history, with empty registry columns.

In [9]:
history_full = history.merge(cad_small, on="CD_CVM", how="left")
history_full["IS_FINANCIAL"] = history_full["SETOR_ATIV"].str.contains(FINANCIAL_KEYWORDS, na=False)

is_active = history_full["SIT"] == "ATIVO"
is_category_a = history_full["CATEG_REG"] == "Categoria A"
is_listed = history_full["TP_MERC"] == "BOLSA"
is_operating = history_full["SIT_EMISSOR"] == "FASE OPERACIONAL"
is_not_financial = ~history_full["IS_FINANCIAL"]

history_full["IS_INVESTABLE"] = (
    is_active & is_category_a & is_listed & is_operating & is_not_financial
)

## 5. Known limitations

- **Capex sign.** A few companies report acquisitions with a positive sign (e.g. Even in 2025). The value may be net of sales or a filing error; these cases should be reviewed by hand.
- **Currency scale.** Some companies seem to report a wrong `ESCALA_MOEDA` (e.g. Padtec in 2025). Values should be sanity-checked against company filings.
- **Power transmission.** Part of the capex of transmission companies may be classified in operating activities, as contract assets, and is not captured by the `6.02` rule.
- **Direct method.** Only the indirect-method cash flow statement (`DFC_MI`) is used; companies that report with the direct method (`DFC_MD`) are missing.
- **IFRS 16 (2019).** Since 2019, lease payments are classified as financing, which raises the operating cash flow of companies with many leased stores. Years before and after 2019 are not fully comparable.
- **Interest paid.** Some companies report interest paid within operating activities. For a free cash flow to the firm, it should be added back.